# Retail Demand Forecasting & Discount Calendar

**Goal:** Forecast daily grocery demand across 5 stores and design an optimal seasonal discount calendar.  
**Dataset:** Retail Store Inventory Forecasting (Kaggle)  
**Models:** Linear Regression · Random Forest · XGBoost (best R²=0.90)

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set(style='whitegrid')
TARGET = 'Demand'
DISCOUNT_LEVELS = [0, 5, 10, 15, 20]

## 2. Load & Filter Data

In [ ]:
df = pd.read_csv('../data/retail_store_inventory.csv')
df = df[df['Category'] == 'Groceries'].copy()
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['Store_ID', 'Product_ID', 'Date']).reset_index(drop=True)
print(f'Shape: {df.shape}')
print(f'Date range: {df["Date"].min().date()} → {df["Date"].max().date()}')
df.head()

## 3. Outlier Analysis (kept for time-series integrity)

In [ ]:
numeric_cols = ['Price', 'Competitor_Price', 'Demand', 'Units_Sold', 'Inventory_Level']

for col in numeric_cols:
    if col not in df.columns:
        continue
    z = np.abs((df[col] - df[col].mean()) / df[col].std())
    z_out = (z > 3).sum()
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr_out = ((df[col] < Q1 - 1.5*(Q3-Q1)) | (df[col] > Q3 + 1.5*(Q3-Q1))).sum()
    print(f'{col:25s} | Z-score: {z_out:4d} | IQR: {iqr_out:4d}')

print('\nOutliers kept — epidemic demand spikes are real events, not noise.')

## 4. Normalisation

In [ ]:
cols_to_scale = [c for c in numeric_cols if c in df.columns]
scaler = MinMaxScaler()
df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])
print('MinMax normalised:', cols_to_scale)
df[cols_to_scale].describe().round(3)

## 5. Exploratory Data Analysis

In [ ]:
# Correlation heatmap
corr = df[cols_to_scale].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# Epidemic effect on demand
if 'Epidemic' in df.columns:
    df['month'] = df['Date'].dt.month
    monthly = df.groupby(['month', 'Epidemic'])['Demand'].mean().unstack()
    monthly.plot(figsize=(10, 4), marker='o')
    plt.title('Monthly Demand by Epidemic Status')
    plt.xlabel('Month')
    plt.ylabel('Avg Demand (normalised)')
    plt.show()
    print('Epidemic periods drive the largest anomalies.')

## 6. Feature Engineering

In [ ]:
# Time features
df['year']           = df['Date'].dt.year
df['month']          = df['Date'].dt.month
df['week']           = df['Date'].dt.isocalendar().week.astype(int)
df['day_of_week']    = df['Date'].dt.dayofweek
df['is_weekend']     = (df['day_of_week'] >= 5).astype(int)
df['is_month_start'] = df['Date'].dt.is_month_start.astype(int)
df['quarter']        = df['Date'].dt.quarter
print('Time features added.')

In [ ]:
# Interpolated_Order — fills zero-order days with estimated daily rate
# Formula: Interpolated_Order = Last_Order_Amount / Days_Since_Last_Order

df = df.sort_values(['Store_ID', 'Product_ID', 'Date'])
df['Last_Order_Amount'] = 0.0
df['Last_Order_Date']   = pd.NaT

for key, group in df.groupby(['Store_ID', 'Product_ID']):
    last_d, last_a = None, 0.0
    for idx in group.index:
        if df.at[idx, 'Units_Ordered'] > 0:
            last_d = df.at[idx, 'Date']
            last_a = df.at[idx, 'Units_Ordered']
        df.at[idx, 'Last_Order_Date']   = last_d
        df.at[idx, 'Last_Order_Amount'] = last_a

df['Last_Order_Date'] = pd.to_datetime(df['Last_Order_Date'])
df['Days_Since_Last_Order'] = (df['Date'] - df['Last_Order_Date']).dt.days.fillna(1).clip(lower=1)
df['Interpolated_Order'] = df['Last_Order_Amount'] / df['Days_Since_Last_Order']
print('Interpolated_Order feature added.')

## 7. Model Training & Evaluation

In [ ]:
def evaluate(y_true, y_pred, label):
    r2   = r2_score(y_true, y_pred)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f'{label:45s} | R²={r2:.4f} | MAE={mae:.4f} | RMSE={rmse:.4f}')
    return r2

# Feature sets
original_features = ['Price','Competitor_Price','Inventory_Level',
                     'Epidemic','Promotion','Discount',
                     'month','week','day_of_week','is_weekend','is_month_start']
set2 = [f for f in original_features if f in df.columns]

X = df[set2].fillna(0)
y = df[TARGET]
split = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

print('--- Linear Regression ---')
lr = LinearRegression().fit(X_train, y_train)
evaluate(y_test, lr.predict(X_test), 'LR | Set_2 | TimeSplit')

print('\n--- Random Forest ---')
rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
evaluate(y_test, rf.predict(X_test), 'RF | Set_2 | TimeSplit')

print('\n--- XGBoost (best config: xgb_3) ---')
params = {'objective':'reg:squarederror','max_depth':6,'eta':0.05,
          'subsample':0.8,'colsample_bytree':0.8,'seed':42}
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest  = xgb.DMatrix(X_test,  label=y_test)
xgb_model = xgb.train(params, dtrain, num_boost_round=500,
                       evals=[(dtest,'val')], early_stopping_rounds=20, verbose_eval=False)
evaluate(y_test, xgb_model.predict(dtest), 'XGB | Set_2 | TimeSplit | xgb_3')

## 8. Discount Calendar

In [ ]:
SEASON_MAP = {12:'Winter',1:'Winter',2:'Winter',
              3:'Spring',4:'Spring',5:'Spring',
              6:'Summer',7:'Summer',8:'Summer',
              9:'Autumn',10:'Autumn',11:'Autumn'}
df['Season'] = df['Date'].dt.month.map(SEASON_MAP)

results = []
product_col = 'Product_ID' if 'Product_ID' in df.columns else None

groups = df.groupby(['Season', product_col] if product_col else ['Season'])
for key, subset in groups:
    season = key[0] if product_col else key
    product = key[1] if product_col else 'All'
    avg_price = subset['Price'].mean() if 'Price' in subset.columns else 1.0

    best_rev, best_disc = -np.inf, 0
    for disc in DISCOUNT_LEVELS:
        row = subset.copy()
        if 'Discount' in row.columns:
            row['Discount'] = disc
        X_s = row[[f for f in set2 if f in row.columns]].fillna(0)
        pred = xgb_model.predict(xgb.DMatrix(X_s.values)).mean()
        rev  = pred * avg_price * (1 - disc / 100)
        if rev > best_rev:
            best_rev, best_disc = rev, disc

    results.append({'Season': season, 'Product': product,
                    'Optimal_Discount_%': best_disc, 'Est_Revenue': round(best_rev, 4)})

calendar = pd.DataFrame(results)
print(calendar.head(20).to_string(index=False))

In [ ]:
# Heatmap calendar
pivot = calendar.pivot(index='Product', columns='Season', values='Optimal_Discount_%')
season_order = [s for s in ['Spring','Summer','Autumn','Winter'] if s in pivot.columns]
pivot = pivot[season_order]

plt.figure(figsize=(10, max(4, len(pivot) * 0.5)))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd', linewidths=0.5,
            cbar_kws={'label': 'Optimal Discount (%)'})
plt.title('Product × Season Optimal Discount Calendar')
plt.tight_layout()
plt.show()

calendar.to_csv('discount_calendar.csv', index=False)
print('Saved: discount_calendar.csv')

## 9. Business Recommendations

1. **Discount Calendar** — Apply season-product optimal discounts automatically to clear perishable stock  
2. **Reorder Point** — `reorder_point = avg_demand × lead_time + safety_stock` using model forecasts  
3. **Bundle Promotions** — Pair low-demand perishables with popular items to reduce waste and increase basket value  
4. **Epidemic Flag** — Monitor external events (epidemics, weather) as the model's largest error driver